# Candidate Dataset Exploration

Run this FIRST after downloading the dataset. Goal: understand the schema,
distribution of skills/titles/experience, and tune scoring weights accordingly.

Steps:
1. Load sample_candidates.json
2. Explore schema fields
3. Check skill distributions
4. Check experience year distributions
5. Look at redrob_signals fields
6. Identify potential honeypot patterns

In [ ]:
import json
import gzip
import pandas as pd
import numpy as np
from collections import Counter
from pathlib import Path

# Load sample candidates (50 records, pretty-printed JSON)
sample_path = Path('../data/raw/sample_candidates.json')
with open(sample_path) as f:
    sample = json.load(f)

print(f'Sample size: {len(sample)}')
print(f'\nTop-level keys in first candidate:')
for k, v in sample[0].items():
    print(f'  {k}: {type(v).__name__} = {repr(v)[:80]}')

In [ ]:
# Inspect redrob_signals schema
for c in sample:
    signals = c.get('redrob_signals', {})
    if signals:
        print('redrob_signals keys found:')
        for k in signals.keys():
            print(f'  {k}: {repr(signals[k])[:60]}')
        break

In [ ]:
# Skills distribution
all_skills = []
for c in sample:
    for s in c.get('skills', []) or []:
        if isinstance(s, str):
            all_skills.append(s.lower())
        elif isinstance(s, dict):
            all_skills.append((s.get('name', '') or '').lower())

skill_counts = Counter(all_skills)
print('Top 30 skills across sample:')
for skill, count in skill_counts.most_common(30):
    print(f'  {skill}: {count}')

In [ ]:
# Experience years distribution
import re
from datetime import datetime

def compute_years(candidate):
    direct = candidate.get('total_experience_years') or candidate.get('years_of_experience')
    if direct:
        try: return float(direct)
        except: pass
    total = 0
    for exp in candidate.get('work_experience', []) or []:
        def parse_year(s):
            if not s: return None
            m = re.search(r'\b(19|20)\d{2}\b', str(s))
            return int(m.group()) if m else None
        s = parse_year(exp.get('start_date') or exp.get('start_year'))
        e = parse_year(exp.get('end_date') or exp.get('end_year')) or datetime.now().year
        if s and e >= s:
            total += (e - s)
    return total

years = [compute_years(c) for c in sample]
df_years = pd.Series(years)
print(df_years.describe())
print('\nDistribution:')
print(df_years.value_counts().sort_index())

In [ ]:
# Title distribution
titles = [
    c.get('current_title', '') or c.get('title', '') or 'UNKNOWN'
    for c in sample
]
print('Current titles:')
for title, count in Counter(titles).most_common(20):
    print(f'  {title}: {count}')

In [ ]:
# Honeypot analysis on sample
import sys
sys.path.append('..')
from src.signals.honeypot import is_honeypot

flagged = []
for c in sample:
    hp, reason = is_honeypot(c)
    if hp:
        flagged.append((c.get('candidate_id', 'unknown'), reason))

print(f'Honeypots detected in sample: {len(flagged)}/{len(sample)}')
for cid, reason in flagged:
    print(f'  {cid}: {reason}')

In [ ]:
# Run mini pipeline on sample to verify end-to-end
from src.ranker.pipeline import RankingPipeline

pipeline = RankingPipeline(top_n=10)
results = pipeline.run(sample)
print('Top 10 from sample:')
print(results[['rank', 'candidate_id', 'score', 'reasoning']].to_string(index=False))